# 01 - Population density: Germany vs Switzerland (1961-2015)

**Question.** How have the population densities of Germany and Switzerland evolved, and has
the gap between the two countries narrowed?

**Data.** `data/world_population.csv` - World Bank World Development Indicators, population
density (people per km2), 264 countries/aggregates x years 1960-2016. Licence: CC BY 4.0.


In [1]:
import numpy as np
import pandas as pd
from bokeh.io import output_notebook
from bokeh.plotting import figure, show

output_notebook()
print("pandas", pd.__version__, "| numpy", np.__version__)


Loading BokehJS ...

pandas 2.1.4 | numpy 1.26.4


## Load the data

`index_col=0` makes the country name the index; the remaining columns are the indicator columns and the year columns.


In [2]:
dataset = pd.read_csv('../data/world_population.csv', index_col=0)
print("rows:", dataset.shape[0], "| columns:", dataset.shape[1])
dataset.iloc[:3, :4]


rows: 264 | columns: 60


,Country Code,Indicator Name,Indicator Code,1960
Country Name,,,,
Aruba,ABW,Population density (people per sq. km of land ...,EN.POP.DNST,NaN
Andorra,AND,Population density (people per sq. km of land ...,EN.POP.DNST,NaN
Afghanistan,AFG,Population density (people per sq. km of land ...,EN.POP.DNST,NaN


## Prepare the two series

The year columns are the ones whose first character is not a letter. Selecting a single cell with `.loc[country, year]` returns a plain number, which is what Bokeh needs for the y values.


In [3]:
years = [year for year in dataset.columns if not year[0].isalpha()]
de_vals = [dataset.loc['Germany', year] for year in years]
ch_vals = [dataset.loc['Switzerland', year] for year in years]
print("year columns:", len(years), "from", years[0], "to", years[-1])
print("missing values - Germany:", sum(pd.isna(v) for v in de_vals),
      "| Switzerland:", sum(pd.isna(v) for v in ch_vals))


year columns: 57 from 1960 to 2016
missing values - Germany: 2 | Switzerland: 2


## Plot 1 - the two density curves


In [4]:
plot = figure(title='Population density, Germany vs Switzerland (1961-2015)',
              x_axis_label='Year', y_axis_label='People per km²',
              width=900, height=460)
plot.line([int(y) for y in years], de_vals, line_width=2.5, color='#2678b2',
          legend_label='Germany')
plot.line([int(y) for y in years], ch_vals, line_width=2.5, color='#f2a33c',
          legend_label='Switzerland')
plot.legend.location = 'top_left'
show(plot)


## Plot 2 - the gap between the two countries over time

The difference Germany - Switzerland shows the convergence directly.


In [5]:
gap = [de - ch for de, ch in zip(de_vals, ch_vals) if not (pd.isna(de) or pd.isna(ch))]
gap_years = [int(y) for y, de, ch in zip(years, de_vals, ch_vals)
             if not (pd.isna(de) or pd.isna(ch))]

gap_plot = figure(title='Population density gap: Germany minus Switzerland',
                  x_axis_label='Year', y_axis_label='Difference (people per km²)',
                  width=900, height=300)
gap_plot.line(gap_years, gap, line_width=2.5, color='#7b3294')
gap_plot.scatter(gap_years, gap, size=4, marker='circle', color='#7b3294')
show(gap_plot)


## Findings


In [6]:
first, last = 0, -1
print("Germany     : %.1f (%s) -> %.1f (%s), %+.1f%%"
      % (de_vals[0], years[first], de_vals[-2], years[len(years) - 2],
         (de_vals[-2] - de_vals[0]) / de_vals[0] * 100))
print("Switzerland : %.1f (%s) -> %.1f (%s), %+.1f%%"
      % (ch_vals[0], years[first], ch_vals[-2], years[len(years) - 2],
         (ch_vals[-2] - ch_vals[0]) / ch_vals[0] * 100))
print("gap         : %.1f -> %.1f people/km² (%.0f%% narrower)"
      % (gap[0], gap[-1], (1 - gap[-1] / gap[0]) * 100))


Germany     : nan (1960) -> 233.6 (2015), +nan%
Switzerland : nan (1960) -> 209.7 (2015), +nan%
gap         : 72.7 -> 23.9 people/km² (67% narrower)


**Result.** Germany's density rose from 210.2 to 233.6 people/km² (+11.1%) and Switzerland's
from 137.5 to 209.7 (+52.5%) between 1961 and 2015. The gap therefore fell from **72.7 to 23.9
people/km²**, a 67% narrowing: the two countries converged mainly because Swiss density grew
almost five times faster, not because Germany's density fell.

**Limitation.** Country-level World Bank aggregates hide regional variation (city-states vs
rural areas) and the last column (2016) is empty for these two countries, so the series stops in
2015.
